PART 1 - multicollinearity; correlation vs causation

**An e-commerce company observes that customers who receive more promotional emails tend to make more purchases.
The company concludes that sending more emails always causes higher sales.**

**Q1- Explain the difference between correlation and causation in this situation.**

*1- More promotional emails and more purchases happen together in the data, showing a link rather than proof of cause.*

*2- Sending more emails directly forces customers to buy more, which requires controlled testing rather than raw observation to prove.*

*3- High-intent buyers or active shoppers may already buy more regardless of emails, or heavy emailing could backfire and cause unsubscribes.*

**Q2- Identify two possible confounding variables.**

*1- Repeat or high-intent shoppers already buy frequently and stay engaged on the list, inflating both email volume and purchases.*

*2- Major events (like holidays or flash sales) trigger spikes in both promotional email sends and customer buying.*

**Q3- Explain why the company's conclusion may be incorrect.**

*1- Existing high-intent buyers or major sales events can boost purchases on their own while aligning with higher email sends.*

*2- Customers who actively shop or browse may trigger or stay open to more emails, rather than emails creating the demand.*

*3- "Always" ignores fatigue, spam flags, and unsubscribes that can lower sales when email frequency gets too high.*

**Q4 - Suggest an experiment to estimate the causal effect of promotional emails on purchases.**

*1- Assign active subscribers randomly into a treatment group (gets the extra promotional emails) and a control group (gets zero extra emails) over a fixed window like 2 to 4 weeks.*

*2- Send identical offers, timing, and store experiences to both groups so email volume is the only difference.*

*3- Measure the difference in conversion rate, order count, or revenue per user between groups to estimate the true causal lift.*

In [15]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

data = {
    "sqft_living": [1250, 1850, 2400, 1520, 3100, 2150, 980, 2750, 1680, 3600],
    "bedrooms": [2, 3, 4, 3, 4, 3, 2, 4, 3, 5],
    "bathrooms": [1.5, 2.0, 2.5, 2.0, 3.5, 2.5, 1.0, 3.0, 2.0, 4.0],
    "lot_sqft": [4200, 6500, 8000, 5000, 10500, 7200, 3100, 9200, 5800, 14000],
    "price": [245000, 340000, 465000, 295000, 620000, 410000, 185000, 540000, 315000, 780000],
}

df = pd.DataFrame(data)


def find_high_corr(df, threshold=0.95):
    corr = df.corr(numeric_only=True)
    cols = corr.columns

    for i in range(len(cols)):
         for j in range(i + 1, len(cols)):
            if corr.iloc[i, j] > 0.95:
                print(f"{cols[i]} - {cols[j]}: {corr.iloc[i, j]:.4f}")

print(find_high_corr(df))

X = df[["sqft_living", "bedrooms", "bathrooms", "lot_sqft"]]

X_with_const = X.copy()
X_with_const["intercept"] = 1
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X.shape[1])]
display(vif_data)

X_reduced = df.drop(columns=["lot_sqft", "bathrooms", "bedrooms"])
print(X_reduced.head())


sqft_living - bedrooms: 0.9562
sqft_living - bathrooms: 0.9890
sqft_living - lot_sqft: 0.9895
sqft_living - price: 0.9945
bedrooms - lot_sqft: 0.9517
bedrooms - price: 0.9521
bathrooms - lot_sqft: 0.9806
bathrooms - price: 0.9864
lot_sqft - price: 0.9960
None


,Feature,VIF
0,sqft_living,93.198769
1,bedrooms,11.957508
2,bathrooms,46.507200
3,lot_sqft,48.997569


   sqft_living   price
0         1250  245000
1         1850  340000
2         2400  465000
3         1520  295000
4         3100  620000


PART 2 -  R²/Adjusted R²; regression assumptions

**A real-estate company builds a Linear Regression model using Area, Bedrooms, Bathrooms, and Age to predict house prices.
The residual plot shows a curved pattern, and prediction errors increase as predicted prices increase.**

**Q1- Identify the assumptions that may be violated.**

*The relationship between the independent variables (Area, Bedrooms, Bathrooms, Age) and the target variable (Price) is non-linear. The linear model fails to capture non-linear relationships or interaction effects such as diminished marginal value per square foot as home size increases.*

**Q2- Explain why a curved residual pattern is important.**

*A curve means the true relationship with features like home size or age is non-linear, so a straight-line model systematically over- or under-predicts prices in specific ranges.*

**Q3- Explain why increasing residual variance can be a problem.**

*Coefficient estimates lose efficiency, and wide error swings at higher price tiers create bigger valuation mistakes on expensive homes.*

**Q4- Suggest appropriate diagnostic plots or statistical tests**

*Show which feature (Area, Bedrooms, Bathrooms, Age) has a non-linear curve.*

*Shows whether error spread grows as predicted price increases.*

*checks for missing non-linear terms (non-linearity).*

PART 3 - feature selection; overfitting; regularization

**A Data Scientist predicts salaries using education, experience, age, skills, and several other features. Some features are highly correlated.**

**Q1- Explain why Regularization may be useful.**

*1- Prevents wildly opposite, large weights when predictors like age and experience move together.*

*2- Penalizes complex weight patterns across many salary and skill features so the model generalizes better.*

*3- Shrinks weak or redundant feature weights (driving some to zero) to keep the model focused and interpretable*

**Q2- Compare Lasso and Ridge Regression.**

*1- Lasso uses an L1 penalty that can shrink weak or redundant feature weights to exact zero; Ridge uses an L2 penalty that shrinks weights smoothly toward zero without eliminating any feature.*

*2- Ridge shares weight across strongly correlated predictors (such as age and experience), keeping them together; Lasso tends to pick one feature from a correlated group and zero out the rest.*

*3- Lasso produces sparser, more interpretable models when only a subset of features truly matters; Ridge keeps all features and often performs better when many small effects contribute together.*

**Q3- Describe when Elastic Net may be appropriate**

*1- Ideal when features are strongly correlated—like age, experience, or overlapping skill indicators—because it can select or shrink whole correlated groups together rather than arbitrarily dropping all but one like Lasso.*

*2- Works well when you have many predictors (such as dozens of skills, tools, and roles) and want sparse feature selection without Lasso's instability on correlated columns.*

*3- Fits scenarios where you want both the feature-elimination behavior of L1 and the weight-shrinking stability of L2, controlled by tuning the mixing ratio and penalty strength.*

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
data = {
    "Area": [
        1500, 2200, 1100, 3100, 1800,
        2500, 950, 2800, 1350, 3500,
        1650, 2100, 1200, 2900, 1750,
        2400, 850, 3200, 1950, 2650,
    ],
    "Bedrooms": [
        3, 4, 2, 5, 3,
        4, 2, 4, 3, 5,
        3, 3, 2, 4, 3,
        4, 1, 4, 3, 4,
    ],
    "Age": [
        10, 5, 25, 2, 15,
        8, 30, 4, 18, 1,
        12, 7, 20, 6, 14,
        9, 35, 3, 11, 5,
    ],
    "Bathrooms": [
        2, 3, 1, 4, 2,
        3, 1, 3, 2, 4,
        2, 2.5, 1.5, 3.5, 2,
        2.5, 1, 3.5, 2, 3,
    ],
    "Random_ID": [f"ID_{i}" for i in range(101, 121)],
    "Price": [
        310000, 480000, 210000, 675000, 340000,
        520000, 175000, 590000, 270000, 750000,
        325000, 440000, 235000, 610000, 350000,
        495000, 150000, 690000, 390000, 560000,
    ],
}

df01 = pd.DataFrame(data)
X1 = df01.drop(columns=["Price", "Random_ID"])
y1 = df01["Price"]
X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

selector = SelectKBest(score_func=f_regression, k=2)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)
selected_features = selector.get_feature_names_out()
print("selected features:", selected_features)

model = LinearRegression()
model.fit(X_train_selected, y_train)
y_pred = model.predict(X_test_selected)
print("y predicted:",y_pred)

test_r2 = r2_score(y_test, y_pred)
print(f"Test R2 score: {test_r2:.2f}")

selected features: ['Area' 'Bathrooms']
y predicted: [301390.25454456 677151.15350106 494464.05525989 463731.30246678]
Test R2 score: 0.99
